# Monte Carlo path tracing of a Cornell box

*graphics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/path_tracing)

Photorealistic rendering solves the rendering equation by Monte Carlo. Every camera ray of every sample is traced as one batch through a box of planes and spheres: cosine-weighted bounces, a shadow ray to the area light at each hit, and Russian roulette to end paths. Watch the noise fall as $1/\sqrt{N}$ and the red and green walls bleed colour onto the white ones, which only global illumination captures.

**Vector form**

$$L(\mathbf x) = L_e(\mathbf x) + \int_{\Omega} \frac{\rho}{\pi}\,L\big(\mathbf x'(\boldsymbol\omega)\big)\,\cos\theta\;d\omega \;\approx\; \frac{1}{N}\sum_{s=1}^{N} \sum_{k} \beta_k\,D_k,\qquad p(\boldsymbol\omega) = \frac{\cos\theta}{\pi}$$

**Index (tensor) form**

$$t_{rp} = \frac{d_p - n_{pi}\,o_{ri}}{n_{pj}\,\omega_{rj}} \qquad \beta_{k+1,rc} = \beta_{krc}\,\rho_{krc} \qquad D_{rc} = \frac{\rho_{rc}}{\pi}\,L^e_c\,\frac{(n_{ri}\,\omega_{ri})(-m_j\,\omega_{rj})\,A}{\lvert \mathbf y - \mathbf x\rvert_r^2}\,V_r \qquad L_{pc} = \frac{1}{N}\sum_s \sum_k \beta_{k,(sp)c}\,D_{k,(sp)c}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $o_{ri},\ \omega_{ri}$ | `camera_rays(cam, spp, gen, device)` | (R, 3) | origin and direction of ray $r = (s, p)$: $N$ jittered samples for each of the $P$ pixels, one batch |
| $n_{pi},\ d_p$ | `scene['n'], scene['d']` | (Pl, 3), (Pl,) | plane $p$ is $n_{pi} x_i = d_p$, clipped to a bounding box; the light is a small plane |
| $t_{rp}$ | `intersect_planes(o, d, scene)` | (R, Pl) | distance from every ray to every plane; repeated $i$, $j$ ⇒ summed |
| $t_{rs}$ | `intersect_spheres(o, d, scene)` | (R, S) | roots of $\lvert o + t\omega - c\rvert^2 = r^2$ for every ray and sphere |
| $\min_p t_{rp}$ | `t_all.min(dim=1)` | (R,) | nearest hit along each ray: which primitive, where, and its normal |
| $\omega \sim \cos\theta/\pi$ | `cosine_sample(n, gen)` | (R, 3) | next direction: uniform point in a disc lifted onto the hemisphere; $f\cos\theta/p = \rho$ |
| $\beta_{krc}$ | `beta = beta * rho` | (R, 3) | path throughput: how much of the light found at bounce $k$ reaches the camera |
| $D_{rc}$ | `direct_light(x, n, rho, scene, gen)` | (R, 3) | next-event estimation: a shadow ray to a random point $y$ on the area light |
| $q_r = \min(0.95, \max_c \beta_{rc})$ | `survive = rand < q;  beta / q` | (R,) | Russian roulette: end dim paths at random but re-weight survivors, so the mean is unchanged |
| $L_{pc}$ | `total.reshape(m, -1, 3).sum(0) / spp` | (P, 3) | the pixel colour: average over the sample index $s$ |

The rendering equation for a diffuse (Lambertian) surface says that light leaving a point is its emission plus the reflected fraction $\rho$ of all incoming light. The incoming light is in turn the outgoing light of other surfaces, so the integral recurses. A path tracer replaces it by random paths. Cosine-weighted sampling ($p = \cos\theta/\pi$) cancels the BRDF and the cosine, so the throughput just picks up a factor $\rho$ per bounce. At every hit a shadow ray to a random point on the light adds the direct light (next-event estimation), which is far less noisy than waiting for a bounce to hit the small light by chance. Only light from the first hit gives the direct-only image. Every further bounce adds indirect light: soft fill in the shadows, a brighter ceiling and red and green colour bleeding. The furnace test in the test suite checks the estimator. A closed box whose walls all emit $L_e$ and reflect $\rho$ must have radiance $L_e/(1-\rho)$ everywhere. With cosine sampling this comes out with zero variance. The error falls as $1/\sqrt N$ because each pixel is a mean of $N$ independent samples. Halving the noise costs four times the samples, which is why production renderers use denoisers.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/path_tracing/main.py
"""Main loop: parameters → Cornell box → progressive render + high-spp reference → noise analysis → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    res = 32 if fast else params["resolution"]
    spp, ref_spp = (16, 64) if fast else (params["spp"], params["reference_spp"])
    roulette = params["termination"] == "roulette"
    scene = tensors.cornell_box(dev, params["light_power"])
    cam = tensors.camera(res)
    n_prims = len(scene["albedo"])
    print(f"Cornell box: {len(scene['n'])} planes + {len(scene['radius'])} spheres = {n_prims} primitives")
    print(f"{res}×{res} pixels, {res * res * 16:,} rays per batch of 16 samples per pixel")
    rule = f"Russian roulette after {params['bounces']} bounces" if roulette else f"{params['bounces']} bounces"
    print(f"paths end by {rule}")

    milestones = sorted({round(v) for v in torch.logspace(0, torch.log10(torch.tensor(spp)), 30).tolist()})
    start = time.perf_counter()
    gi, di = solve.progressive(scene, cam, milestones, torch.Generator(dev).manual_seed(1), params["bounces"], roulette)
    t_prog = time.perf_counter() - start
    start = time.perf_counter()
    ref_gi, ref_di = solve.render(scene, cam, ref_spp, torch.Generator(dev).manual_seed(2), params["bounces"], roulette)
    t_ref = time.perf_counter() - start
    print(f"progressive render to {spp} spp: {t_prog:.1f} s; reference with {ref_spp} spp: {t_ref:.1f} s")

    ns = torch.tensor(milestones, dtype=torch.float32)
    err_gi, err_di = solve.rmse(gi, ref_gi).cpu(), solve.rmse(di, ref_di).cpu()
    usable = ns <= spp / 4  # the reference has noise of its own; fit where it is still small by comparison
    slope = solve.noise_slope(ns[usable], err_gi[usable])
    for i in sorted({0, len(milestones) // 3, 2 * len(milestones) // 3, len(milestones) - 1}):
        print(f"  spp {milestones[i]:4d}: RMSE vs reference {err_gi[i]:.4f} (GI), {err_di[i]:.4f} (direct only)")
    print(f"log-log slope of RMSE against spp: {slope:.2f} (Monte Carlo: −0.5, i.e. noise ∝ 1/√spp)")

    left, right = tensors.pixel_of(cam, [-0.8, 0.3, -1.0]), tensors.pixel_of(cam, [0.8, 0.3, -1.0])
    for name, (r, c) in (("left (next to red)", left), ("right (next to green)", right)):
        g, d = ref_gi[r - 1 : r + 2, c - 1 : c + 2].mean((0, 1)), ref_di[r - 1 : r + 2, c - 1 : c + 2].mean((0, 1))
        print(f"white back wall, {name}: red/green = {g[0] / g[1]:.2f} with GI, {d[0] / d[1]:.2f} direct only")
    share = 1 - ref_di.mean() / ref_gi.mean()
    print(f"indirect (bounced) light is {100 * share:.0f}% of the image's total brightness")
    return plot.animate(gi, di, ref_gi, ref_di, milestones, err_gi, err_di, ref_spp, out_dir / "path_tracing.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/path_tracing/equation.py
r"""Rendering equation → tensor translation.

The light leaving a diffuse surface point x towards the eye is the emitted light plus all incoming light
reflected with the Lambertian BRDF f = ρ/π:

    L(x) = L_e(x) + ∫_{hemisphere} (ρ/π) L(x'(ω)) cos θ dω,

where x'(ω) is the first surface hit looking from x in direction ω. A path tracer estimates this integral by
Monte Carlo, for a batch of rays r at once. Sampling ω with the cosine-weighted pdf p(ω) = cos θ / π cancels
the cosine and the 1/π, so every bounce just multiplies the path throughput by the albedo:

    β_{k+1, rc} = β_{krc} ρ_{krc}          (c = colour channel)

Direct light from the area light (corner, edges e₁, e₂, area A, normal m) is added at every bounce by
sampling a point y on it (next-event estimation):

    D_{rc} = (ρ_{rc}/π) L^e_c (n_{ri} ω_{ri}) (−m_i ω_{ri}) A / |y − x|²_r · V_r       (i summed)

with V_r = 1 if nothing blocks the shadow ray. The pixel value averages L_{rc} = Σ_k β_{krc} D_{krc} (+ L_e
seen directly) over N samples, so its noise falls as 1/√N.
"""

import math

import torch

EPS = 1e-4


def intersect_planes(o: torch.Tensor, d: torch.Tensor, scene: dict) -> torch.Tensor:
    r"""t_{rp} = (d_p − n_{pi} o_{ri}) / (n_{pj} d_{rj})   (i, j summed), +∞ where the hit is behind or off the wall.

    o, d (R, 3) → (R, Pl)"""
    denom = torch.einsum("pj,rj->rp", scene["n"], d)
    t = (scene["d"] - torch.einsum("pi,ri->rp", scene["n"], o)) / denom
    x = o[:, None] + t[..., None] * d[:, None]  # (R, Pl, 3) hit points
    inside = ((x >= scene["lo"] - EPS) & (x <= scene["hi"] + EPS)).all(-1)
    return torch.where((t > EPS) & inside & (denom.abs() > 1e-9), t, math.inf)


def intersect_spheres(o: torch.Tensor, d: torch.Tensor, scene: dict) -> torch.Tensor:
    r"""|o + t d − c|² = r²  ⇒  t = −b ∓ √(b² − c), b = d_i (o − c)_i, c = (o − c)_i (o − c)_i − r².   → (R, S)"""
    oc = o[:, None] - scene["centre"]  # (R, S, 3)
    b = torch.einsum("ri,rsi->rs", d, oc)
    c = torch.einsum("rsi,rsi->rs", oc, oc) - scene["radius"] ** 2
    disc = b**2 - c
    root = disc.clamp(min=0).sqrt()
    near, far = -b - root, -b + root
    t = torch.where(near > EPS, near, far)  # from inside a sphere only the far root counts
    return torch.where((disc > 0) & (t > EPS), t, math.inf)


def nearest_hit(o: torch.Tensor, d: torch.Tensor, scene: dict):
    """Closest primitive along every ray: distance t (R,), primitive id (R,) (−1 = escaped) and the unit normal
    facing the incoming ray (R, 3)."""
    t_all = torch.cat([intersect_planes(o, d, scene), intersect_spheres(o, d, scene)], dim=1)  # (R, Pl + S)
    t, prim = t_all.min(dim=1)
    n_planes = scene["n"].shape[0]
    x = o + t.clamp(max=1e6)[:, None] * d
    plane_n = scene["n"][prim.clamp(max=n_planes - 1)]
    s = (prim - n_planes).clamp(min=0, max=max(0, len(scene["radius"]) - 1))
    sphere_n = (x - scene["centre"][s]) / scene["radius"][s, None] if len(scene["radius"]) else plane_n
    normal = torch.where((prim < n_planes)[:, None], plane_n, sphere_n)
    normal = torch.where((normal * d).sum(-1, keepdim=True) > 0, -normal, normal)  # face the ray
    return t, torch.where(torch.isfinite(t), prim, -1), normal


def cosine_sample(n: torch.Tensor, gen: torch.Generator) -> torch.Tensor:
    r"""Directions with pdf p(ω) = cos θ / π about the normals n (R, 3) → (R, 3).

    Pick a uniform point in the unit disc (radius √u₁, angle 2π u₂) and lift it onto the hemisphere."""
    u = torch.rand(n.shape[0], 2, generator=gen, device=n.device)
    r, phi = u[:, 0].sqrt(), 2 * math.pi * u[:, 1]
    local = torch.stack([r * torch.cos(phi), r * torch.sin(phi), (1 - u[:, 0]).clamp(min=0).sqrt()], -1)
    helper = torch.where(n[:, :1].abs() > 0.9, n.new_tensor([0.0, 1, 0]), n.new_tensor([1.0, 0, 0]))
    t1 = torch.linalg.cross(helper, n)
    t1 = t1 / t1.norm(dim=1, keepdim=True)
    t2 = torch.linalg.cross(n, t1)
    frame = torch.stack([t1, t2, n], dim=1)  # (R, 3, 3): rows = tangent, bitangent, normal
    return torch.einsum("ra,rai->ri", local, frame)  # ω_i = local_a · frame_{ai}  (a summed)


def cosine_pdf(cos_theta: torch.Tensor) -> torch.Tensor:
    """p(ω) = cos θ / π on the upper hemisphere, 0 below."""
    return cos_theta.clamp(min=0) / math.pi


def direct_light(x: torch.Tensor, n: torch.Tensor, rho: torch.Tensor, scene: dict, gen: torch.Generator):
    r"""Next-event estimate D_{rc} = (ρ/π) L^e (n·ω)(−m·ω) A / |y − x|² · V with y uniform on the light.  → (R, 3)"""
    light = scene["light"]
    u = torch.rand(x.shape[0], 2, generator=gen, device=x.device)
    y = light["corner"] + u[:, :1] * light["e1"] + u[:, 1:] * light["e2"]  # (R, 3) point on the light
    w = y - x
    dist2 = (w * w).sum(-1)
    omega = w / dist2.sqrt()[:, None]
    cos_x = torch.einsum("ri,ri->r", n, omega).clamp(min=0)
    cos_y = -torch.einsum("i,ri->r", light["normal"], omega).clamp(max=0)  # the light faces −m
    area = torch.linalg.cross(light["e1"], light["e2"]).norm()
    t_block, _, _ = nearest_hit(x + EPS * n, omega, scene)
    visible = t_block >= dist2.sqrt() * (1 - 1e-3)  # the first thing the shadow ray meets is the light itself
    g = cos_x * cos_y * area / dist2 * visible
    return rho / math.pi * light["Le"] * g[:, None]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/path_tracing/tensors.py
"""Build the concrete tensors: the scene (bounded planes, spheres, materials, an area light) and the camera rays.

Planes are stored as n_{pi} x_i = d_p with an axis-aligned bounding box [lo, hi] (so a plane is a wall, not an
infinite sheet); spheres as centre and radius. Primitive ids run over the planes first, then the spheres.
"""

import torch

WHITE, RED, GREEN = [0.75, 0.75, 0.75], [0.70, 0.12, 0.10], [0.15, 0.60, 0.15]
BIG = 10.0  # "unbounded" along the plane's own normal direction


def _plane(normal, d, lo, hi, albedo, emission=(0.0, 0.0, 0.0)):
    return normal, d, lo, hi, albedo, emission


def build(planes: list, spheres: list, light: dict | None, device: torch.device) -> dict:
    """Stack the primitive lists into tensors. spheres: (centre, radius, albedo)."""

    def t(rows):
        return torch.tensor(rows, dtype=torch.float32, device=device)

    scene = {
        "n": t([p[0] for p in planes]),  # (Pl, 3) unit normals
        "d": t([p[1] for p in planes]),  # (Pl,) offsets: n·x = d on the plane
        "lo": t([p[2] for p in planes]),  # (Pl, 3) bounding box of each wall
        "hi": t([p[3] for p in planes]),
        "centre": t([s[0] for s in spheres]).reshape(-1, 3),  # (S, 3)
        "radius": t([s[1] for s in spheres]).reshape(-1),  # (S,)
        "albedo": t([p[4] for p in planes] + [s[2] for s in spheres]),  # (Pl + S, 3) diffuse reflectance ρ
        "emission": t([p[5] for p in planes] + [[0.0, 0.0, 0.0] for _ in spheres]),  # (Pl + S, 3) emitted L_e
        "light": None,
    }
    if light is not None:
        scene["light"] = {k: t(v) for k, v in light.items()}
    return scene


def cornell_box(device: torch.device, light_power: float = 12.0) -> dict:
    """The room [−1, 1]³ open towards the camera (+z): white floor, ceiling and back wall, red left wall, green
    right wall, a 0.6 × 0.6 square light just below the ceiling and two white spheres."""
    e = BIG
    le = [light_power] * 3
    planes = [
        _plane([0, 1, 0], -1.0, [-1, -e, -1], [1, e, 1], WHITE),  # floor y = −1
        _plane([0, -1, 0], -1.0, [-1, -e, -1], [1, e, 1], WHITE),  # ceiling y = 1
        _plane([0, 0, 1], -1.0, [-1, -1, -e], [1, 1, e], WHITE),  # back wall z = −1
        _plane([1, 0, 0], -1.0, [-e, -1, -1], [e, 1, 1], RED),  # left wall x = −1
        _plane([-1, 0, 0], -1.0, [-e, -1, -1], [e, 1, 1], GREEN),  # right wall x = 1
        _plane([0, -1, 0], -0.99, [-0.3, -e, -0.3], [0.3, e, 0.3], [0, 0, 0], le),  # the area light at y = 0.99
    ]
    spheres = [([-0.42, -0.6, -0.35], 0.4, WHITE), ([0.45, -0.68, 0.3], 0.32, WHITE)]
    # The light for next-event estimation: a corner, two edge vectors, its normal and its radiance.
    light = {"corner": [-0.3, 0.99, -0.3], "e1": [0.6, 0, 0], "e2": [0, 0, 0.6], "normal": [0, -1, 0], "Le": le}
    return build(planes, spheres, light, device)


def furnace(device: torch.device, albedo: float = 0.5, emission: float = 1.0) -> dict:
    """A closed box whose six walls all emit L_e and reflect a fraction ρ: the radiance everywhere is L_e/(1 − ρ)."""
    e, a, le = BIG, [albedo] * 3, [emission] * 3
    planes = [
        _plane([0, 1, 0], -1.0, [-1, -e, -1], [1, e, 1], a, le),
        _plane([0, -1, 0], -1.0, [-1, -e, -1], [1, e, 1], a, le),
        _plane([0, 0, 1], -1.0, [-1, -1, -e], [1, 1, e], a, le),
        _plane([0, 0, -1], -1.0, [-1, -1, -e], [1, 1, e], a, le),
        _plane([1, 0, 0], -1.0, [-e, -1, -1], [e, 1, 1], a, le),
        _plane([-1, 0, 0], -1.0, [-e, -1, -1], [e, 1, 1], a, le),
    ]
    return build(planes, [], None, device)


def camera(res: int, eye: tuple = (0.0, 0.0, 3.4), focal: float = 2.3) -> dict:
    """Pinhole camera looking down −z; focal length in units of the half image width."""
    return {"eye": torch.tensor(eye), "focal": focal, "res": res}


def camera_rays(cam: dict, spp: int, gen: torch.Generator, device: torch.device):
    """spp jittered rays through every pixel: origins and unit directions of shape (spp·res·res, 3).

    Ray r = s·P + p (sample s, pixel p in row-major order from the top-left)."""
    res = cam["res"]
    s = torch.arange(res, device=device, dtype=torch.float32)
    yy, xx = torch.meshgrid(s, s, indexing="ij")
    jitter = torch.rand(spp, res, res, 2, generator=gen, device=device)  # random point inside each pixel
    x = (2 * (xx + jitter[..., 0]) / res - 1) / cam["focal"]
    y = (1 - 2 * (yy + jitter[..., 1]) / res) / cam["focal"]
    d = torch.stack([x, y, -torch.ones_like(x)], -1).reshape(-1, 3)
    d = d / d.norm(dim=1, keepdim=True)
    return cam["eye"].to(device).expand_as(d), d


def pixel_of(cam: dict, point: list) -> tuple[int, int]:
    """(row, column) of the pixel that sees a world point (used to probe the image)."""
    p = torch.tensor(point) - cam["eye"]
    x, y = cam["focal"] * p[0] / -p[2], cam["focal"] * p[1] / -p[2]
    res = cam["res"]
    return int((1 - y) * res / 2), int((x + 1) * res / 2)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/path_tracing/solve.py
"""The path tracer: follow every ray of a batch bounce by bounce, then average samples into images."""

import torch

from problems._common.progress import steps

from .equation import EPS, cosine_sample, direct_light, nearest_hit
from .tensors import camera_rays


def trace(o: torch.Tensor, d: torch.Tensor, scene: dict, gen: torch.Generator, bounces: int, roulette: bool):
    """Radiance carried back along each ray: global illumination and direct-only, both (R, 3).

    Only live paths are kept in the batch (escaped or terminated ones are dropped), and their contributions are
    scattered back to the original ray index with index_add_. With roulette, after `bounces` bounces a path survives
    with probability q = max_c β_c (≤ 0.95) and is re-weighted by 1/q, which keeps the estimate unbiased; otherwise
    it stops after exactly `bounces` bounces."""
    n_rays = o.shape[0]
    total = torch.zeros(n_rays, 3, device=o.device)
    direct = torch.zeros(n_rays, 3, device=o.device)
    idx = torch.arange(n_rays, device=o.device)  # original ray of every live path
    beta = torch.ones(n_rays, 3, device=o.device)  # path throughput β
    nee = scene["light"] is not None
    max_depth = 4 * bounces + 8 if roulette else bounces
    for k in range(max_depth + 1):
        t, prim, n = nearest_hit(o, d, scene)
        live = prim >= 0  # paths that escape through the open front are dropped
        idx, o, d, t, prim, n, beta = idx[live], o[live], d[live], t[live], prim[live], n[live], beta[live]
        if len(idx) == 0:
            break
        x = o + t[:, None] * d
        rho = scene["albedo"][prim]
        contrib = torch.zeros_like(beta)
        if k == 0 or not nee:  # with next-event estimation, light reached by bouncing is already counted
            contrib += beta * scene["emission"][prim]
        if nee:
            contrib += beta * direct_light(x, n, rho, scene, gen)
        total.index_add_(0, idx, contrib)
        if k == 0:
            direct.index_add_(0, idx, contrib)
        if k == max_depth:
            break
        beta = beta * rho  # f cos θ / p = (ρ/π) cos θ / (cos θ/π) = ρ
        if roulette and k + 1 >= bounces:
            q = beta.max(dim=1).values.clamp(max=0.95)
            survive = torch.rand(len(idx), generator=gen, device=o.device) < q
            beta = beta / q.clamp(min=1e-6)[:, None]
            idx, x, n, beta = idx[survive], x[survive], n[survive], beta[survive]
        o, d = x + EPS * n, cosine_sample(n, gen)
    return total, direct


def render(scene, cam, spp: int, gen, bounces: int, roulette: bool, chunk: int = 16):
    """Mean of spp samples per pixel: GI and direct images (res, res, 3), rendered chunk samples at a time."""
    res, dev = cam["res"], scene["n"].device
    gi = torch.zeros(res * res, 3, device=dev)
    di = torch.zeros(res * res, 3, device=dev)
    for s in steps((spp + chunk - 1) // chunk, "sample batches"):
        m = min(chunk, spp - s * chunk)
        o, d = camera_rays(cam, m, gen, dev)
        total, direct = trace(o, d, scene, gen, bounces, roulette)
        gi += total.reshape(m, -1, 3).sum(0)
        di += direct.reshape(m, -1, 3).sum(0)
    return (gi / spp).reshape(res, res, 3), (di / spp).reshape(res, res, 3)


def progressive(scene, cam, milestones: list[int], gen, bounces: int, roulette: bool, chunk: int = 16):
    """Keep adding samples and snapshot the running mean at each milestone: (M, res, res, 3) for GI and direct."""
    res, dev = cam["res"], scene["n"].device
    gi_sum = torch.zeros(res, res, 3, device=dev)
    di_sum = torch.zeros(res, res, 3, device=dev)
    gi_snaps, di_snaps, done = [], [], 0
    for m in steps(len(milestones), "milestones"):
        while done < milestones[m]:
            n = min(chunk, milestones[m] - done)
            gi, di = render(scene, cam, n, gen, bounces, roulette, chunk)
            gi_sum, di_sum, done = gi_sum + n * gi, di_sum + n * di, done + n
        gi_snaps.append(gi_sum / done)
        di_snaps.append(di_sum / done)
    return torch.stack(gi_snaps), torch.stack(di_snaps)


def rmse(images: torch.Tensor, reference: torch.Tensor) -> torch.Tensor:
    """Root-mean-square error against the reference in the displayed range [0, 1]: (M, H, W, 3), (H, W, 3) → (M,).

    Clipping first keeps the few pixels on the edge of the bright light (radiance 0 or L_e at random) from
    dominating the average."""
    return ((images.clamp(0, 1) - reference.clamp(0, 1)) ** 2).mean(dim=(-3, -2, -1)).sqrt()


def noise_slope(spp: torch.Tensor, err: torch.Tensor) -> float:
    """Least-squares slope of log(error) against log(spp); Monte Carlo predicts −1/2."""
    x, y = spp.log(), err.log()
    x, y = x - x.mean(), y - y.mean()
    return float((x * y).sum() / (x * x).sum())


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/path_tracing/plot.py
"""Progressive render (GI and direct only), the indirect light of the reference, and the noise against spp."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def tonemap(img: torch.Tensor) -> np.ndarray:
    """Linear radiance → displayable sRGB-like values (clip, then gamma 2.2)."""
    return img.clamp(0, 1).pow(1 / 2.2).cpu().numpy()


def animate(gi, di, ref_gi, ref_di, milestones, err_gi, err_di, ref_spp, path: Path) -> Path:
    """gi, di (M, H, W, 3) running means at the milestone sample counts; ref_* (H, W, 3)."""
    ns = np.array(milestones)
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.6))
    (ax_gi, ax_di), (ax_ind, ax_err) = axes
    im_gi = ax_gi.imshow(tonemap(gi[0]), interpolation="nearest")
    im_di = ax_di.imshow(tonemap(di[0]), interpolation="nearest")
    indirect = (ref_gi - ref_di).clamp(min=0)
    ax_ind.imshow(tonemap(indirect * 3), interpolation="nearest")
    ax_ind.set_title(f"indirect light only (×3), {ref_spp} spp", fontsize=10)
    for ax in (ax_gi, ax_di, ax_ind):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)

    ax_err.loglog(ns, err_gi.numpy(), color="#2563eb", lw=1.5, label="global illumination")
    ax_err.loglog(ns, err_di.numpy(), color="#ea580c", lw=1.5, label="direct only")
    ax_err.loglog(ns, err_gi[0].item() / np.sqrt(ns), "--", color="#9ca3af", lw=1.2, label=r"$\propto 1/\sqrt{spp}$")
    (dot,) = ax_err.plot([], [], "o", color="#2563eb", ms=6)
    ax_err.set_xlabel("samples per pixel")
    ax_err.set_ylabel("RMSE vs reference")
    ax_err.set_title("noise", fontsize=10)
    ax_err.legend(fontsize=8, loc="lower left")

    def update(m: int):
        im_gi.set_data(tonemap(gi[m]))
        im_di.set_data(tonemap(di[m]))
        ax_gi.set_title(f"path traced (GI), {ns[m]} spp", fontsize=10)
        ax_di.set_title(f"direct light only, {ns[m]} spp", fontsize=10)
        dot.set_data([ns[m]], [err_gi[m].item()])

    update(0)
    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(ns), 120), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'resolution': 96,
    'spp': 128,
    'reference_spp': 512,
    'bounces': 3,
    'termination': 'roulette',
    'light_power': 30.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.path_tracing")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.path_tracing.main")
problem_dir = Path("problems/graphics/path_tracing")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
